# synthkit worked example: Bike Sharing

Chosen for two things the other three examples don't have. First, a genuine datetime column
(`dteday`), `DatetimeMarginal`'s path through the copula had only ever been exercised by
synthetic unit-test data before this dataset, and testing against it caught a real bug (a
unit mismatch in the fidelity check between real and synthetic dates).

Second, an exact derived-column relationship on real data: `cnt` is always precisely
`casual + registered`, every single row, a real-world justification for the `Derived`
constraint, not a synthetic example invented to demonstrate it.

In [1]:
import subprocess
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

import synthkit as sk

DATA_DIR = Path("../data")
ZIP_PATH = DATA_DIR / "bike-sharing.zip"
CSV_PATH = DATA_DIR / "day.csv"
DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00275/Bike-Sharing-Dataset.zip"

DATA_DIR.mkdir(parents=True, exist_ok=True)
if not CSV_PATH.exists():
    subprocess.run(["curl", "-sL", "-o", str(ZIP_PATH), DATA_URL], check=True)
    with zipfile.ZipFile(ZIP_PATH) as archive:
        archive.extract("day.csv", DATA_DIR)

df = pd.read_csv(CSV_PATH)
df["dteday"] = pd.to_datetime(df["dteday"])
df = df.drop(columns=["instant"])
df.head()

,dteday,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,2011-01-01,1,0,1,0,6,0,2,0.344167,0.363625,0.805833,0.160446,331,654,985
1,2011-01-02,1,0,1,0,0,0,2,0.363478,0.353739,0.696087,0.248539,131,670,801
2,2011-01-03,1,0,1,0,1,1,1,0.196364,0.189405,0.437273,0.248309,120,1229,1349
3,2011-01-04,1,0,1,0,2,1,1,0.200000,0.212122,0.590435,0.160296,108,1454,1562
4,2011-01-05,1,0,1,0,3,1,1,0.226957,0.229270,0.436957,0.186900,82,1518,1600


In [2]:
profile = sk.fit(df)
print("dteday classified as:", profile.column_types["dteday"])
print("fitted granularity (seconds):", profile.marginals["dteday"]["granularity_seconds"])

dteday classified as: datetime
fitted granularity (seconds): 86400


## Does the date correlate with rental count, before and after?

In [3]:
synthetic = sk.emit(profile, n=len(df), seed=0)

real_epoch = df["dteday"].to_numpy().astype("datetime64[s]").astype("float64")
synth_epoch = synthetic["dteday"].to_numpy().astype("datetime64[s]").astype("float64")

real_corr = np.corrcoef(real_epoch, df["cnt"])[0, 1]
synth_corr = np.corrcoef(synth_epoch, synthetic["cnt"].astype(float))[0, 1]
print(f"corr(date, rental count): real={real_corr:.3f} synthetic={synth_corr:.3f}")

corr(date, rental count): real=0.629 synthetic=0.555


## A real derived-column relationship

`cnt` is always exactly `casual + registered`. The copula models all three as
separately-correlated numeric columns, so without declaring the relationship, it only holds
by chance.

In [4]:
exact_real = (df["casual"] + df["registered"] == df["cnt"]).mean()
exact_synth_no_constraint = (
    synthetic["casual"] + synthetic["registered"] == synthetic["cnt"]
).mean()
print(f"cnt == casual + registered, real data:                 {exact_real:.3f}")
print(f"cnt == casual + registered, synthetic (no constraint): {exact_synth_no_constraint:.3f}")

cnt == casual + registered, real data:                 1.000
cnt == casual + registered, synthetic (no constraint): 0.001


In [5]:
profile_with_constraint = sk.fit(df, constraints=[sk.Derived("cnt", "casual + registered")])
synthetic_with_constraint = sk.emit(profile_with_constraint, n=len(df), seed=0)

exact_synth_with_constraint = (
    synthetic_with_constraint["casual"] + synthetic_with_constraint["registered"]
    == synthetic_with_constraint["cnt"]
).mean()
label = "cnt == casual + registered, synthetic (Derived constraint)"
print(f"{label}: {exact_synth_with_constraint:.3f}")

cnt == casual + registered, synthetic (Derived constraint): 1.000


## Privacy check

In [6]:
report = sk.check(synthetic, profile, real=df, min_dcr_ratio=0.5)
print(f"dcr_ratio: {report.dcr_ratio:.3f}")
print(f"exact_matches: {report.exact_matches}")

dcr_ratio: 2.941
exact_matches: 0
